# 01 — NWB schema audit

Opens the selected NWB with PyNWB when possible and writes schema manifest/report artifacts.

In [1]:
import sys
import time
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from _omission_run_common import (
    base_manifest,
    build_schema_report,
    project_root,
    render_schema_report_md,
    resolve_nwb_path,
    resolve_run_root,
    resolve_nwb_sha256,
    run_nwb_audit,
    write_json,
    write_warnings,
)

start = time.time()
repo = project_root()
nwb_path = resolve_nwb_path(repo)
nwb_sha256 = resolve_nwb_sha256(nwb_path)
run_root = resolve_run_root(repo, nwb_path, nwb_sha256)
run_root.mkdir(parents=True, exist_ok=True)

warnings = []
report, open_error = run_nwb_audit(nwb_path, build_schema_report)
if open_error:
    warnings.append({"code": "PYNWB_OPEN_FAILED", "message": open_error})
    report = {
        "summary": "Schema audit blocked: PyNWB could not open the selected NWB.",
        "pynwb_open_status": "failed",
        "error": open_error,
    }

manifest = base_manifest(
    notebook_id="01_nwb_schema_audit",
    analysis_stage="schema_audit",
    warnings_rel="warnings/01_warnings.json",
    runtime_seconds=time.time() - start,
    repo=repo,
    nwb_path=nwb_path,
    run_root=run_root,
    nwb_sha256=nwb_sha256,
    outputs=[
        "manifests/nwb_schema_manifest.json",
        "reports/nwb_schema_report.json",
        "reports/nwb_schema_report.md",
        "warnings/01_warnings.json",
    ],
)
manifest["pynwb_open_status"] = report.get("pynwb_open_status", "failed")

write_json(run_root / "manifests" / "nwb_schema_manifest.json", manifest)
write_json(run_root / "reports" / "nwb_schema_report.json", report)
(run_root / "reports" / "nwb_schema_report.md").write_text(
    render_schema_report_md(report) if open_error is None else (
        "# NWB Schema Audit Report\n\n"
        f"- pynwb_open_status: failed\n"
        f"- error: {open_error}\n"
    ),
    encoding="utf-8",
)
write_warnings(run_root / "warnings" / "01_warnings.json", warnings)

status = "PASS" if open_error is None else "BLOCKED"
print(f"Schema audit status: {status}")
print("run_root:", run_root)
if open_error:
    raise RuntimeError(f"Notebook 01 blocked: {open_error}")

Schema audit status: PASS
run_root: D:\workspace\omission\outputs\runs\52461b8e06890033c93c6dbfb2453a4699a732c2_17e93e3f
